# Advanced 12 — Capstone: Secure, Compliant & Resilient Enterprise Agent Identity Platform

This notebook integrates the full Agent Identity curriculum into one enterprise claims platform.

**Architecture rule:** the LLM is an untrusted planner. Identity, delegation, authorization, approval, credentials, telemetry and governance are trusted runtime/control-plane responsibilities.


In [ ]:
from dataclasses import dataclass,field
from datetime import datetime,timedelta,timezone
import copy,hashlib,hmac,json,secrets,uuid
import pandas as pd, networkx as nx
NOW=datetime.now(timezone.utc)


## 1 — Define platform invariants

In [ ]:
INVARIANTS=[
"model_identity_untrusted","approved_workload_binding","no_static_prod_secret",
"audience_binding","token_exchange_attenuation","delegation_attenuation",
"cross_tenant_deny","pep_coverage","deterministic_authz","high_risk_step_up",
"no_mcp_token_passthrough","federated_auth_local_authz","revocation_invalidates_cache",
"no_raw_tokens_in_telemetry","critical_action_evidence","quarantine_breaks_attack_path"]
pd.DataFrame({"invariant":INVARIANTS})

## 2 — Register agents

In [ ]:
registry={
"agent:claims":{"registered":True,"owner":"claims-ai","risk":"high","status":"active"},
"agent:research":{"registered":True,"owner":"claims-ai","risk":"medium","status":"active"}}
registry

## 3 — Lifecycle state machine

In [ ]:
allowed={"draft":{"pending_approval"},"pending_approval":{"approved","revoked"},
"approved":{"active","revoked"},"active":{"suspended","quarantined","retired","revoked"},
"suspended":{"active","retired","revoked"},"quarantined":{"suspended","revoked"},
"retired":{"revoked"},"revoked":set()}
def transition(a,b):
    if b not in allowed[a]:raise ValueError(f"blocked {a}->{b}")
    return b
try: transition("draft","active")
except ValueError as e: print(e)

## 4 — Bind logical agent to workload

In [ ]:
workload={"spiffe_id":"spiffe://corp.example/prod/claims-orchestrator","agent_id":"agent:claims","approved":True}
assert workload["approved"] and workload["agent_id"]=="agent:claims"

## 5 — Build trusted security context

In [ ]:
@dataclass(frozen=True)
class SecurityContext:
    principal_id:str;tenant_id:str;agent_id:str;workload_id:str;task_id:str;delegation_id:str
ctx=SecurityContext("user:alice","acme","agent:claims",workload["spiffe_id"],"task:483","dlg:483")
ctx

## 6 — Issue short-lived workload-derived token

In [ ]:
def issue(sub,aud,scope,minutes=10,actor=None):
    return {"jti":secrets.token_hex(8),"sub":sub,"act":actor,"aud":aud,"scope":set(scope),
            "exp":NOW+timedelta(minutes=minutes)}
runtime_token=issue("agent:claims","token-broker",{"exchange"},5)
runtime_token

## 7 — Token exchange with attenuation

In [ ]:
def exchange(parent,aud,scope,minutes=5):
    if not set(scope).issubset(parent["scope"]|{"claim.read","claim.update","knowledge.search"}):
        raise PermissionError("scope escalation")
    return issue(parent["sub"],aud,set(scope),minutes,parent["sub"])
claims_token=exchange(runtime_token,"claims-api",{"claim.read","claim.update"},5)
claims_token

## 8 — Audience validation

In [ ]:
def audience_ok(t,aud):return t["aud"]==aud and NOW<t["exp"]
audience_ok(claims_token,"claims-api"),audience_ok(claims_token,"payments-api")

## 9 — Delegation root

In [ ]:
delegation={"delegator":"user:alice","delegatee":"agent:claims","tenant":"acme",
"actions":{"claim.read","claim.update","knowledge.search"},"resources":{"claim:483","kb:claims"},
"expires_at":NOW+timedelta(hours=1),"depth":0,"max_depth":1,"revoked":False}
delegation

## 10 — Create attenuated child delegation

In [ ]:
def child_delegation(parent,delegatee,actions,resources,minutes):
    child={"delegator":parent["delegatee"],"delegatee":delegatee,"tenant":parent["tenant"],
    "actions":set(actions),"resources":set(resources),"expires_at":NOW+timedelta(minutes=minutes),
    "depth":parent["depth"]+1,"max_depth":parent["max_depth"],"revoked":False}
    assert child["actions"].issubset(parent["actions"])
    assert child["resources"].issubset(parent["resources"])
    assert child["expires_at"]<=parent["expires_at"]
    assert child["depth"]<=parent["max_depth"]
    return child
child=child_delegation(delegation,"agent:research",{"claim.read","knowledge.search"},{"claim:483","kb:claims"},15)
child

## 11 — Relationship model

In [ ]:
relationships={
("user:alice","assigned_to","claim:483"):True,
("agent:claims","assigned_to","task:483"):True,
("agent:research","can_invoke","mcp:policy-search"):True}
relationships

## 12 — Typed intent

In [ ]:
@dataclass
class Intent:
    action:str;resource:str;tool:str;purpose:str;parameters:dict=field(default_factory=dict)
intent=Intent("claim.update","claim:483","claims.update","process assigned claim",{"status":"reviewed"})
intent

## 13 — Rich authorization decision

In [ ]:
@dataclass
class Decision:
    outcome:str;decision_id:str;reason:str;constraints:dict=field(default_factory=dict);obligations:list=field(default_factory=list)


## 14 — Hybrid PDP

In [ ]:
def authorize(ctx,i,d,risk="low"):
    deny=lambda r:Decision("deny",uuid.uuid4().hex,r)
    if d["tenant"]!=ctx.tenant_id:return deny("TENANT")
    if d["delegatee"]!=ctx.agent_id:return deny("DELEGATEE")
    if i.action not in d["actions"]:return deny("ACTION")
    if i.resource not in d["resources"]:return deny("RESOURCE")
    if relationships.get((ctx.principal_id,"assigned_to",i.resource)) is not True:return deny("RELATIONSHIP")
    if risk=="critical":return deny("CRITICAL_RISK")
    if risk=="high":return Decision("step_up",uuid.uuid4().hex,"HIGH_RISK",{},["human_approval","audit"])
    constraints={"allowed_fields":{"status","notes"}} if i.action=="claim.update" else {}
    return Decision("allow",uuid.uuid4().hex,"TASK_SCOPE",constraints,["audit"])
decision=authorize(ctx,intent,delegation)
decision

## 15 — PEP constraint enforcement

In [ ]:
def enforce(i,d):
    if d.outcome!="allow":raise PermissionError(d.reason)
    allowed=d.constraints.get("allowed_fields")
    if allowed is not None and not set(i.parameters).issubset(allowed):
        raise PermissionError("FIELD_CONSTRAINT")
    return "executed"
enforce(intent,decision)

## 16 — Cross-tenant attack

In [ ]:
evil=copy.deepcopy(delegation);evil["tenant"]="other"
authorize(ctx,intent,evil)

## 17 — RAG authorization

In [ ]:
docs=[
{"id":"d1","tenant":"acme","acl":{"claims"}},
{"id":"d2","tenant":"other","acl":{"claims"}},
{"id":"d3","tenant":"acme","acl":{"hr"}}]
groups={"claims"}
[d for d in docs if d["tenant"]==ctx.tenant_id and d["acl"]&groups]

## 18 — Memory authorization

In [ ]:
memory=[{"id":"m1","tenant":"acme","owner":"user:alice"},{"id":"m2","tenant":"acme","owner":"user:bob"}]
[m for m in memory if m["tenant"]==ctx.tenant_id and m["owner"]==ctx.principal_id]

## 19 — MCP resource binding

In [ ]:
mcp_token=issue("agent:research","https://mcp.claims.example",{"knowledge.search"},5)
assert mcp_token["aud"]=="https://mcp.claims.example"
print("Do not reuse at https://claims-api.internal")

## 20 — Federated partner identity

In [ ]:
foreign={"spiffe_id":"spiffe://partner.example/prod/research-agent","trust_domain":"partner.example","verified":True}
trust_registry={"partner.example":{"active":True,"allowed_actions":{"knowledge.search"}}}
foreign["verified"] and trust_registry[foreign["trust_domain"]]["active"]

## 21 — Federated local authorization

In [ ]:
requested="knowledge.search"
foreign_allow=(foreign["verified"] and requested in trust_registry["partner.example"]["allowed_actions"]
               and requested in child["actions"])
foreign_allow

## 22 — High-risk payment intent

In [ ]:
payment=Intent("payment.create","claim:483","payments.create","settle claim",{"amount":750,"currency":"CAD"})
authorize(ctx,payment,delegation,"high")

## 23 — Transaction-bound approval

In [ ]:
def digest(i):
    body={"agent":ctx.agent_id,"task":ctx.task_id,"action":i.action,"resource":i.resource,"parameters":i.parameters}
    return hashlib.sha256(json.dumps(body,sort_keys=True,separators=(",",":")).encode()).hexdigest()
approval={"approver":"user:manager","digest":digest(payment),"expires":NOW+timedelta(minutes=5),"used":False}
approval

## 24 — Approval parameter swap attack

In [ ]:
changed=copy.deepcopy(payment);changed.parameters["amount"]=7500
digest(changed)==approval["digest"]

## 25 — One-use payment capability

In [ ]:
payment_cap={"action":"payment.create","resource":"claim:483","max_amount":750,"currency":"CAD",
"expires":NOW+timedelta(minutes=5),"used":False}
def use_payment(cap,amount,currency):
    if cap["used"] or NOW>=cap["expires"] or amount>cap["max_amount"] or currency!=cap["currency"]:return False
    cap["used"]=True;return True
use_payment(payment_cap,700,"CAD"),use_payment(payment_cap,700,"CAD")

## 26 — Identity telemetry

In [ ]:
def evt(kind,actor,**kw):
    return {"schema":"agent.identity.event/1.0","event_type":kind,"timestamp":NOW.isoformat(),
            "trace_id":"trace-483","actor":actor,**kw}
events=[
evt("agent.invoke","user:alice",resource="agent:claims"),
evt("delegation.issued","agent:claims",resource="agent:research",delegation_id="dlg:child"),
evt("authorization.decision","agent:claims",action="claim.update",resource="claim:483",decision="allow",policy_version="v17"),
evt("tool.execute","agent:claims",action="claim.update",resource="claims-api")]
pd.DataFrame(events)

## 27 — No raw token logging

In [ ]:
log={"actor":"agent:claims","access_token":"ey-secret","credential_fingerprint":"fp-991"}
safe={k:("[REDACTED]" if k=="access_token" else v) for k,v in log.items()}
safe

## 28 — Build identity graph

In [ ]:
G=nx.DiGraph()
G.add_edges_from([
("user:alice","agent:claims"),("agent:claims","agent:research"),("agent:research","mcp:policy-search"),
("mcp:policy-search","claims-api"),("claims-api","role:claims"),("role:claims","kms:evidence")])
list(G.edges())

## 29 — Attack path

In [ ]:
nx.shortest_path(G,"user:alice","kms:evidence")

## 30 — Stolen token detection

In [ ]:
stolen={"credential":"fp-991","aud":"claims-api","used_at":"payments-api"}
finding="audience_anomaly" if stolen["aud"]!=stolen["used_at"] else None
finding

## 31 — Delegation escalation detection

In [ ]:
bad_child=copy.deepcopy(child);bad_child["actions"].add("payment.create")
"delegation_escalation" if not bad_child["actions"].issubset(delegation["actions"]) else None

## 32 — Quarantine

In [ ]:
agent_state={"agent:research":"active"}
agent_state["agent:research"]="quarantined"
agent_state

## 33 — Revoke token and delegation

In [ ]:
revoked_tokens={mcp_token["jti"]}
child["revoked"]=True
{"token_revoked":mcp_token["jti"] in revoked_tokens,"delegation_revoked":child["revoked"]}

## 34 — Invalidate authorization cache

In [ ]:
cache={("agent:research","knowledge.search","kb:claims"):"ALLOW"}
cache.clear()
cache

## 35 — Verify attack path reduction

In [ ]:
G2=G.copy()
G2.remove_edge("agent:claims","agent:research")
nx.has_path(G2,"agent:research","kms:evidence"), nx.has_path(G2,"user:alice","kms:evidence")

## 36 — Recovery gate

In [ ]:
recovery={"persistence_removed":True,"keys_rotated":True,"workload_reattested":True,
"delegations_reviewed":True,"telemetry_healthy":True,"policy_validated":True}
all(recovery.values())

## 37 — Continuous compliance checks

In [ ]:
agents=[
{"id":"agent:claims","owner":"claims-ai","environment":"prod","credential":"workload","review_age":20,"monitoring":True},
{"id":"agent:legacy","owner":"","environment":"prod","credential":"static_api_key","review_age":200,"monitoring":False}]
def compliance(a):
    f=[]
    if not a["owner"]:f.append("missing_owner")
    if a["environment"]=="prod" and a["credential"]=="static_api_key":f.append("static_prod_credential")
    if a["review_age"]>90:f.append("review_overdue")
    if not a["monitoring"]:f.append("monitoring_missing")
    return f
[(a["id"],compliance(a)) for a in agents]

## 38 — Critical override

In [ ]:
raw_score=92
critical=["static_prod_credential"]
effective=min(raw_score,25) if critical else raw_score
effective

## 39 — Evidence chain

In [ ]:
def canon(x):return json.dumps(x,sort_keys=True,separators=(",",":"))
def chain(rows):
    out=[];prev=""
    for r in rows:
        h=hashlib.sha256((prev+canon(r)).encode()).hexdigest()
        out.append({"event":r,"prev":prev,"hash":h});prev=h
    return out
evidence_chain=chain(events)
evidence_chain[-1]

## 40 — Evidence verification

In [ ]:
def verify(c):
    prev=""
    for row in c:
        if row["prev"]!=prev:return False
        if hashlib.sha256((prev+canon(row["event"])).encode()).hexdigest()!=row["hash"]:return False
        prev=row["hash"]
    return True
verify(evidence_chain)

## 41 — Tamper evidence test

In [ ]:
tampered=copy.deepcopy(evidence_chain)
tampered[1]["event"]["resource"]="agent:payment-admin"
verify(tampered)

## 42 — Audit pack

In [ ]:
audit_pack={"scope":"claims agent platform","period":"2026-Q3",
"agents":[a["id"] for a in agents],"critical_findings":critical,
"evidence_root":evidence_chain[-1]["hash"],"policy_version":"v17","generated_at":NOW.isoformat()}
audit_pack

## 43 — Production scorecard

In [ ]:
scorecard=pd.DataFrame([
["Agent registry","PASS"],["Workload binding","PASS"],["Token attenuation","PASS"],
["Delegation attenuation","PASS"],["Cross-tenant isolation","PASS"],["PEP coverage","PASS"],
["MCP token isolation","PASS"],["Federated local authz","PASS"],["HITL binding","PASS"],
["Observability","PASS"],["ITDR","PASS"],["Compliance","FAIL: legacy identity"]],columns=["control","status"])
scorecard

## 44 — Final architecture review

In [ ]:
review_questions=[
"identity","authentication","credential","delegation","resource","PDP","PEP",
"failure behavior","revocation","telemetry","evidence","adversarial test"]
pd.DataFrame({"required_answer":review_questions})

# Final capstone challenge

You now have a complete teaching platform.

Extend it so the following scenario is provably safe:

```text
Alice
  ↓
Claims Orchestrator
  ↓
Research Agent
  ↓
External Partner Research Agent
  ↓
MCP Policy Search
  ↓
Claims API

and:

Claims Orchestrator
  ↓
STEP-UP
  ↓
Human Approval
  ↓
One-use Payment Capability
  ↓
Payments API
```

Then simulate a compromise of `agent:research`.

Your final submission should include:

- identity inventory;
- trust boundaries;
- runtime/workload bindings;
- token and delegation model;
- OPA/Cedar/OpenFGA policies;
- AuthZEN-style request/decision examples;
- MCP Protected Resource Metadata;
- federation/trust registry;
- trace/evidence schema;
- attack-path graph;
- detection rules;
- quarantine/recovery runbook;
- compliance scorecard;
- audit pack;
- final architecture review.

**Success means the platform can explain and prove every critical action, not merely execute it.**
